# Exploratory Data Analysis (EDA)

**Auteur**: `Jakob De Vreese`  
**Bachelorproef**: Overdraagbare unsupervised anomaliedetectie bij hybride HVAC-systemen  
**Academiejaar**: `2025-2026`  

## Doel

In deze notebook analyseren we de opgehaalde _gebouwdata_ om de **datakzaliteit**, **operationele patronen** en **bruikbaarheid** voor modellering te beoordelen.
De focus ligt op kortetermijn- en operationele inwichtenm aangezien de beschikbare periode slechts enkele maanden omvat en ds niet geschikt is voor robuuste seizoensanalyse.

### Doel van de EDA

- Begrijpen hoe de HVAC-installatie zich gedraagt in de tijd.
- Datakwaliteitsproblemen dedecteren vooraleer modellering start.
- Relevante features en subsystemen selecteren voor anomaly detection.
- Baseline-gedrag en afwijkende patronen zichtaar maken via visualisaties.

## Opbouw van de notebook

- **Sectie 1**: Databegrip en structuur
- **Sectie 2**: Datakwaliteit en opschoning
- **Sectie 3**: Univariate tijdreeksanalyse
- **Sectie 4**: Multivariate relaties en systeemgedrag
- **Sectie 5**: Operationele segmentatie
- **Sectie 6**: Voorbereiding op modellering

### Verwachte output van de EDA

- Gekwalificeerde, opgeschoonde en gedocumenteerde dataset
- Visualisaties die normaal gedrag en afwijkende patronen duidelijk maken

In [64]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from pandas.api import types as ptypes

## Sectie 0: inlezen van de dataset

In [65]:
# Hyperparameters
GEBOUWNAAM = 'dunant1'

In [66]:
# Inlezen dataset
url = f'../01_dataverzameling/preprocessed/{GEBOUWNAAM}.csv'
data = pd.read_csv(url, parse_dates=['timestamp'], index_col='timestamp').sort_index()

data.head()

,f_1,f_2,f_3,f_4,f_6,f_7,f_8,f_9,f_10,f_11,...,f_46,f_47,f_48,f_49,f_50,f_51,f_52,f_53,f_54,f_55
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-03-09 00:10:00+00:00,7.0,0.77,1.0,39.877285,31.832268,1.0,23.594538,44.640636,1.0,22.222490,...,50.0,19.666667,20.75229,487.712690,0.0,3.851797,99.589400,0.749325,86.946641,0.0
2026-03-09 00:20:00+00:00,9.0,0.78,1.0,41.921010,34.104786,1.0,46.930084,44.640636,1.0,38.731342,...,50.0,19.666667,20.75229,485.292560,0.0,3.903977,100.000000,0.559565,45.771835,0.0
2026-03-09 00:30:00+00:00,8.0,0.86,1.0,43.412975,35.585346,1.0,43.235270,44.640636,0.0,41.083800,...,50.0,19.666667,20.75229,486.288320,0.0,4.244891,99.196013,0.322963,91.228401,0.0
2026-03-09 00:40:00+00:00,7.0,0.78,1.0,43.734930,35.924030,1.0,48.845810,44.640636,0.0,46.879260,...,50.0,19.666667,20.75229,490.234433,0.0,4.448537,98.800579,0.300000,323.796930,0.0
2026-03-09 00:50:00+00:00,9.0,0.78,1.0,44.037724,36.229050,1.0,44.068630,44.640636,0.0,44.223957,...,50.0,19.666667,20.75229,488.682280,0.0,4.748848,99.259927,0.345344,204.933704,0.0


### Dictionary met semantische namen per functioneel punt construeren

Aan de hand van de bronnenlijst van het gebouw construeren we een dictionary zodat we in de verdere notebook altijd terug kunnen grijpen naar de namen van de punten

In [67]:
url_bronnen = f'../01_dataverzameling/bronbestanden/bronnen-{GEBOUWNAAM}.csv'

bronnen = pd.read_csv(
    url_bronnen,
    usecols=["func_dp_nr", "func_dp_naam", "func_systeem"]
)

f_map = (
    bronnen
    .assign(
        kolom=lambda d: "f_" + d["func_dp_nr"].astype(str),
        label=lambda d: d["func_dp_naam"].str.strip() + " - " + d["func_systeem"].str.strip()
    )
    .groupby("kolom")["label"]
    .agg(lambda s: " | ".join(pd.unique(s)))   # vangt dubbele func_dp_nr netjes op
    .to_dict()
)

f_map

{'f_1': 'totaal warmtegebruik warm - warmtepomp(en)',
 'f_10': 'status - gasketel(s)',
 'f_11': 'temperatuur retour - gasketel(s)',
 'f_12': 'status pomp - gasketel(s)',
 'f_13': 'totaal warmtegebruik warm injectie - gasketel(s)',
 'f_14': 'totaal debiet injectie - gasketel(s)',
 'f_15': 'stand klep injectie - gasketel(s)',
 'f_16': 'totaal warmtegebruik warm - collector',
 'f_17': 'totaal debiet - collector',
 'f_18': 'temperatuur vertrek - collector',
 'f_19': 'temperatuur retour - collector',
 'f_2': 'totaal debiet - warmtepomp(en)',
 'f_20': 'status pomp - collector',
 'f_21': 'gevraagde temperatuur pulsielucht - luchtgroep 1',
 'f_22': 'temperatuur pulsielucht - luchtgroep 1',
 'f_23': 'temperatuur extractielucht - luchtgroep 1',
 'f_24': 'druk of debiet pulsiekanaal - luchtgroep 1',
 'f_25': 'druk of debiet extractiekanaal - luchtgroep 1',
 'f_26': 'temperatuur verse lucht - luchtgroep 1',
 'f_27': 'sturing pulsieventilator - luchtgroep 1',
 'f_28': 'sturing extractieventilator -

Voeg de omgevingsnamen toe aan de dictionary

In [68]:
omgeving_map = {
  'f_51': 'Buitentemperatuur - omgeving',
  'f_52': 'Luchtvochtigheid - omgeving',
  'f_53': 'Windsnelheid - omgeving',
  'f_54': 'Windrichting - omgeving',
  'f_55': 'Zonnestraling - omgeving'
}

for k, v in omgeving_map.items():
  f_map.setdefault(k, v)


In [69]:
f_map

{'f_1': 'totaal warmtegebruik warm - warmtepomp(en)',
 'f_10': 'status - gasketel(s)',
 'f_11': 'temperatuur retour - gasketel(s)',
 'f_12': 'status pomp - gasketel(s)',
 'f_13': 'totaal warmtegebruik warm injectie - gasketel(s)',
 'f_14': 'totaal debiet injectie - gasketel(s)',
 'f_15': 'stand klep injectie - gasketel(s)',
 'f_16': 'totaal warmtegebruik warm - collector',
 'f_17': 'totaal debiet - collector',
 'f_18': 'temperatuur vertrek - collector',
 'f_19': 'temperatuur retour - collector',
 'f_2': 'totaal debiet - warmtepomp(en)',
 'f_20': 'status pomp - collector',
 'f_21': 'gevraagde temperatuur pulsielucht - luchtgroep 1',
 'f_22': 'temperatuur pulsielucht - luchtgroep 1',
 'f_23': 'temperatuur extractielucht - luchtgroep 1',
 'f_24': 'druk of debiet pulsiekanaal - luchtgroep 1',
 'f_25': 'druk of debiet extractiekanaal - luchtgroep 1',
 'f_26': 'temperatuur verse lucht - luchtgroep 1',
 'f_27': 'sturing pulsieventilator - luchtgroep 1',
 'f_28': 'sturing extractieventilator -

### Metadata en kwaliteitssignalen

Centrale tabel met feature-metadata en datakwaliteitsindicatoren.

In [70]:
# 1) Basis metadata
overzicht = pd.DataFrame({"feature": data.columns})
overzicht["label_raw"] = overzicht["feature"].map(f_map)

# Neem enkel eerste label als er meerdere labels staan (gescheiden door " | ")
overzicht["beschrijving_eerste"] = overzicht["label_raw"].fillna("onbekend").str.split(r"\s\|\s", n=1, regex=True).str[0]

# Splits laatste " - " in [beschrijving, systeem]
split_cols = overzicht["beschrijving_eerste"].str.rsplit(" - ", n=1, expand=True).reindex(columns=[0, 1])
overzicht["beschrijving"] = split_cols[0].fillna("onbekend")
overzicht["systeem"] = split_cols[1].fillna("onbekend")

overzicht.head(10)

,feature,label_raw,beschrijving_eerste,beschrijving,systeem
0,f_1,totaal warmtegebruik warm - warmtepomp(en),totaal warmtegebruik warm - warmtepomp(en),totaal warmtegebruik warm,warmtepomp(en)
1,f_2,totaal debiet - warmtepomp(en),totaal debiet - warmtepomp(en),totaal debiet,warmtepomp(en)
2,f_3,status - warmtepomp(en),status - warmtepomp(en),status,warmtepomp(en)
3,f_4,temperatuur vertrek - warmtepomp(en),temperatuur vertrek - warmtepomp(en),temperatuur vertrek,warmtepomp(en)
4,f_6,temperatuur retour - warmtepomp(en),temperatuur retour - warmtepomp(en),temperatuur retour,warmtepomp(en)
5,f_7,status pomp - warmtepomp(en),status pomp - warmtepomp(en),status pomp,warmtepomp(en)
6,f_8,temperatuur vertrek - gasketel(s),temperatuur vertrek - gasketel(s),temperatuur vertrek,gasketel(s)
7,f_9,gevraagde temperatuur - gasketel(s),gevraagde temperatuur - gasketel(s),gevraagde temperatuur,gasketel(s)
8,f_10,status - gasketel(s),status - gasketel(s),status,gasketel(s)
9,f_11,temperatuur retour - gasketel(s),temperatuur retour - gasketel(s),temperatuur retour,gasketel(s)


In [71]:
# 2) Eenvoudige statistieken
n_obs = len(data)
n_missing = data.isna().sum()
n_unique = data.nunique(dropna=True)
min_v = data.min()
max_v = data.max()
mean_v = data.mean()
std_v = data.std()

In [72]:
# 3) Simpele signaaltypes: binair / discreet / continu
is_binary = data.apply(lambda s: s.dropna().nunique() <= 2 and set(s.dropna().unique()).issubset({0, 1}))
is_integer_like = data.apply(lambda s: np.allclose(s.dropna(), np.round(s.dropna())))
signaal_type = pd.Series(
    np.select(
        [
            is_binary,
            is_integer_like & (n_unique <= 20)
        ],
        [
            "binair",
            "discreet"
        ],
        default="continu"
    ),
    index=data.columns
)

In [73]:
# 4) Finale tabel met exact de kolommen die je vroeg
metadata_stats = pd.DataFrame({
    "feature": data.columns,
    "beschrijving": data.columns.map(overzicht.set_index("feature")["beschrijving"]),
    "systeem": data.columns.map(overzicht.set_index("feature")["systeem"]),
    "n_obs": n_obs,
    "n_missing": data.columns.map(n_missing),
    "n_unique": data.columns.map(n_unique),
    "min": data.columns.map(min_v),
    "max": data.columns.map(max_v),
    "mean": data.columns.map(mean_v),
    "std": data.columns.map(std_v),
    "signaal_type": data.columns.map(signaal_type)
})

metadata_stats = metadata_stats.sort_values(["systeem", "feature"]).reset_index(drop=True)
metadata_stats.head(20)

,feature,beschrijving,systeem,n_obs,n_missing,n_unique,min,max,mean,std,signaal_type
0,f_16,totaal warmtegebruik warm,collector,6765,0,980,0.0,3266.500000,133.504065,271.281226,continu
1,f_17,totaal debiet,collector,6765,0,290,0.0,0.995000,0.109167,0.185225,continu
2,f_18,temperatuur vertrek,collector,6765,0,4001,0.0,47.226585,34.084712,7.017825,continu
3,f_19,temperatuur retour,collector,6765,0,4301,0.0,45.439180,30.570894,6.431179,continu
4,f_20,status pomp,collector,6765,0,2,0.0,1.000000,0.384479,0.486508,binair
5,f_10,status,gasketel(s),6765,0,2,0.0,1.000000,0.152993,0.360008,binair
6,f_11,temperatuur retour,gasketel(s),6765,0,4972,0.0,58.157787,29.645150,7.853388,continu
7,f_12,status pomp,gasketel(s),6765,0,2,0.0,1.000000,0.258980,0.438107,binair
8,f_13,totaal warmtegebruik warm injectie,gasketel(s),6765,0,7,0.0,6.000000,0.024538,0.257911,discreet
9,f_14,totaal debiet injectie,gasketel(s),6765,0,44,0.0,1.000000,0.001905,0.022274,continu


-----

## Sectie 1: Databegrip en structuur

1. Overzicht van beschikbare meetpunten
2. Beschrijving van variabelen per punt
3. Controle van tijdsresolutie, meetfrequentie en andere
4. Eerste verkenning van schaal, eenheden en bereik per signaal

### Sectie 1.1: overzicht van beschikbare meetpunten

In [74]:
print(f"Aantal features: {len(metadata_stats)}")
print(f"Aantal tijdstappen: {len(data)}")

print("\nVerdeling per signaaltype:")
print(metadata_stats["signaal_type"].value_counts(dropna=False))

overzicht_per_systeem = (
metadata_stats
.groupby("systeem", dropna=False)
.agg(
n_features=("feature", "count"),
avg_missing=("n_missing", "mean"),
)
.sort_values("n_features", ascending=False)
)
overzicht_per_systeem

Aantal features: 54
Aantal tijdstappen: 6765

Verdeling per signaaltype:
signaal_type
continu     46
binair       5
discreet     3
Name: count, dtype: int64


,n_features,avg_missing
systeem,,
luchtgroep 2,11,0.0
luchtgroep 1,11,0.0
gasketel(s),8,0.0
warmtepomp(en),6,0.0
collector,5,0.0
omgeving,5,0.0
zone 1,4,0.0
zone 2,4,0.0


### Sectie 1.2: beschrijving van variabelen per punt

In [75]:
metadata_stats

,feature,beschrijving,systeem,n_obs,n_missing,n_unique,min,max,mean,std,signaal_type
0,f_16,totaal warmtegebruik warm,collector,6765,0,980,0.000000,3266.500000,133.504065,271.281226,continu
1,f_17,totaal debiet,collector,6765,0,290,0.000000,0.995000,0.109167,0.185225,continu
2,f_18,temperatuur vertrek,collector,6765,0,4001,0.000000,47.226585,34.084712,7.017825,continu
3,f_19,temperatuur retour,collector,6765,0,4301,0.000000,45.439180,30.570894,6.431179,continu
4,f_20,status pomp,collector,6765,0,2,0.000000,1.000000,0.384479,0.486508,binair
5,f_10,status,gasketel(s),6765,0,2,0.000000,1.000000,0.152993,0.360008,binair
6,f_11,temperatuur retour,gasketel(s),6765,0,4972,0.000000,58.157787,29.645150,7.853388,continu
7,f_12,status pomp,gasketel(s),6765,0,2,0.000000,1.000000,0.258980,0.438107,binair
8,f_13,totaal warmtegebruik warm injectie,gasketel(s),6765,0,7,0.000000,6.000000,0.024538,0.257911,discreet
9,f_14,totaal debiet injectie,gasketel(s),6765,0,44,0.000000,1.000000,0.001905,0.022274,continu


### Sectie 1.3: controle van tijdsresolutie, meetfreauentie en andere

In [76]:
print("Index type:", type(data.index))
print("Start:", data.index.min())
print("Einde:", data.index.max())
print("Tijdzone:", data.index.tz)

n_dupes = data.index.duplicated().sum()
print("Aantal dubbele timestamps:", n_dupes)

diffs = data.index.to_series().diff().dropna()
freq_mode = diffs.mode().iloc[0]
print("Verwachte tijdstap:", freq_mode)

volledige_index = pd.date_range(
start=data.index.min(),
end=data.index.max(),
freq=freq_mode,
tz=data.index.tz
)

ontbrekende_timestamps = volledige_index.difference(data.index)
dekking_pct = 100 * (1 - len(ontbrekende_timestamps) / len(volledige_index))

print(f"Aantal verwachte timestamps: {len(volledige_index)}")
print(f"Aantal ontbrekende timestamps: {len(ontbrekende_timestamps)}")
print(f"Dekking: {dekking_pct:.2f}%")

ontbrekende_timestamps[:10]

Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
Start: 2026-03-09 00:10:00+00:00
Einde: 2026-04-24 23:50:00+00:00
Tijdzone: UTC
Aantal dubbele timestamps: 0
Verwachte tijdstap: 0 days 00:10:00
Aantal verwachte timestamps: 6767
Aantal ontbrekende timestamps: 2
Dekking: 99.97%


DatetimeIndex(['2026-03-30 22:10:00+00:00', '2026-04-14 22:10:00+00:00'], dtype='datetime64[ns, UTC]', freq=None)

### Sectie 1.4: Eerste verkenning van schaal, eenheden en bereik per signaal

In [77]:
stats_s1 = metadata_stats.copy()
stats_s1["range"] = stats_s1["max"] - stats_s1["min"]
stats_s1["cv"] = stats_s1["std"] / (stats_s1["mean"].abs() + 1e-8)

print("Top 15 grootste spreiding (std):")
stats_s1.sort_values("std", ascending=False)[
["feature", "beschrijving", "systeem", "signaal_type", "min", "max", "mean", "std", "range", "cv"]
].head(15)

Top 15 grootste spreiding (std):


,feature,beschrijving,systeem,signaal_type,min,max,mean,std,range,cv
16,f_24,druk of debiet pulsiekanaal,luchtgroep 1,continu,0.000000,3028.578000,1191.358123,871.980703,3028.578000,0.731922
39,f_55,Zonnestraling,omgeving,continu,0.000000,972.466667,223.064429,291.832435,972.466667,1.308288
0,f_16,totaal warmtegebruik warm,collector,continu,0.000000,3266.500000,133.504065,271.281226,3266.500000,2.032007
17,f_25,druk of debiet extractiekanaal,luchtgroep 1,continu,0.000000,264.710600,103.111067,120.537533,264.710600,1.169007
27,f_35,Druk of debiet pulsiekanaal,luchtgroep 2,continu,0.000000,279.112760,89.093618,116.929233,279.112760,1.312431
28,f_36,druk of debiet extractiekanaal,luchtgroep 2,continu,0.000000,307.191600,89.604035,116.506204,307.191600,1.300234
52,f_49,ruimteco2,zone 2,continu,0.000000,1038.431367,478.711154,102.810865,1038.431367,0.214766
38,f_54,Windrichting,omgeving,continu,0.037482,359.916158,222.781644,93.747231,359.878676,0.420803
48,f_45,ruimteco2,zone 1,continu,0.000000,712.318643,512.258612,75.060413,712.318643,0.146528
20,f_28,sturing extractieventilator,luchtgroep 1,continu,0.000000,100.000000,25.559925,31.247762,100.000000,1.222529


------------

## Sectie 2: Datakwaliteit en Opschoning

1. Ontbrekende waarden, gaten in tijdreeksen en dubbele timestamps
2. Detectie van onrealistische waarden en sensorfouten
3. Analyse van statusvelden en binaire signalen op consistentie
4. Harmonisatie en voorbereiding van data voor downstream processing

### Sectie 2.1: Ontbrekende waarden, gaten in tijdreeksen en dubbele timestamps

In [78]:
# Feature-level missings rechtstreeks uit metadata_stats
missings_overzicht = (
    metadata_stats.loc[metadata_stats["n_missing"] > 0,
                       ["feature", "beschrijving", "systeem", "signaal_type", "n_missing", "n_obs"]]
    .assign(missing_pct=lambda d: 100 * d["n_missing"] / d["n_obs"])
    .sort_values("missing_pct", ascending=False)
)

print(f"Aantal features met missings: {len(missings_overzicht)}")
display(missings_overzicht.head(20))

# Globale missing ratio
totaal_missings = metadata_stats["n_missing"].sum()
totaal_cellen = len(data) * data.shape[1]
print(f"Totaal missings: {totaal_missings} ({100 * totaal_missings / totaal_cellen:.2f}%)")

# Tijdsgaten en dubbele timestamps
n_dupes = data.index.duplicated().sum()
diffs = data.index.to_series().diff().dropna()
freq_mode = diffs.mode().iloc[0]

volledige_index = pd.date_range(
    start=data.index.min(),
    end=data.index.max(),
    freq=freq_mode,
    tz=data.index.tz
)
ontbrekende_timestamps = volledige_index.difference(data.index)

print(f"Aantal dubbele timestamps: {n_dupes}")
print(f"Verwachte tijdstap: {freq_mode}")
print(f"Aantal ontbrekende timestamps: {len(ontbrekende_timestamps)}")
display(ontbrekende_timestamps[:10])

Aantal features met missings: 0


,feature,beschrijving,systeem,signaal_type,n_missing,n_obs,missing_pct


Totaal missings: 0 (0.00%)
Aantal dubbele timestamps: 0
Verwachte tijdstap: 0 days 00:10:00
Aantal ontbrekende timestamps: 2


DatetimeIndex(['2026-03-30 22:10:00+00:00', '2026-04-14 22:10:00+00:00'], dtype='datetime64[ns, UTC]', freq=None)

### Missende data opvullen

In [79]:
# missende featurewaarden opvullen (zonder tijdstappen toe te voegen)
data_feat_filled = data.copy()
data_feat_filled = data_feat_filled.interpolate(method="time").ffill().bfill()

print("Resterende missings na feature-opvulling:", int(data_feat_filled.isna().sum().sum()))

Resterende missings na feature-opvulling: 0


In [80]:
# missende tijdstappen opvullen
d = data_feat_filled[~data_feat_filled.index.duplicated(keep="first")].sort_index()
freq = d.index.to_series().diff().mode().iloc[0]
full_index = pd.date_range(d.index.min(), d.index.max(), freq=freq, tz=d.index.tz)

data_time_filled = d.reindex(full_index).interpolate(method="time").ffill().bfill()
data_time_filled.index.name = "timestamp"

print("Nieuwe shape:", data_time_filled.shape)
print("Resterende missings:", int(data_time_filled.isna().sum().sum()))

Nieuwe shape: (6767, 54)
Resterende missings: 0


### Sectie 2.2: Detectie van onrealistische waarden en sensorfouten

In [81]:
# 2.2a - Sparse outliers detecteren op ALLE features
df_num = data_time_filled.copy() if "data_time_filled" in globals() else data.copy()
df_num = df_num[~df_num.index.duplicated(keep="first")].sort_index()

# Globaal robuust (tegen hele reeks)
med = df_num.median()
mad = (df_num - med).abs().median().replace(0, np.nan)
z_global = 0.6745 * (df_num - med).abs().div(mad)

# Lokaal robuust (tegen buurtgedrag, vangt "plots piekje")
roll_med = df_num.rolling(7, center=True, min_periods=3).median()
roll_mad = (df_num - roll_med).abs().rolling(7, center=True, min_periods=3).median().replace(0, np.nan)
z_local = 0.6745 * (df_num - roll_med).abs().div(roll_mad)

# Enkel echte extreme, geïsoleerde pieken
spike_mask = (z_global > 8) & (z_local > 8)

spike_n = spike_mask.sum()
spike_pct = 100 * spike_n / len(df_num)

outlier_overzicht = (
    metadata_stats[["feature", "beschrijving", "systeem", "signaal_type"]]
    .assign(
        spike_n=lambda d: d["feature"].map(spike_n).fillna(0).astype(int),
        spike_pct=lambda d: d["feature"].map(spike_pct).fillna(0.0),
    )
    .sort_values(["spike_pct", "spike_n"], ascending=False)
)

display(outlier_overzicht.head(30))

,feature,beschrijving,systeem,signaal_type,spike_n,spike_pct
16,f_24,druk of debiet pulsiekanaal,luchtgroep 1,continu,93,1.374317
28,f_36,druk of debiet extractiekanaal,luchtgroep 2,continu,85,1.256096
32,f_40,stand klep verwarmingsbatterij,luchtgroep 2,continu,83,1.226541
27,f_35,Druk of debiet pulsiekanaal,luchtgroep 2,continu,68,1.004877
17,f_25,druk of debiet extractiekanaal,luchtgroep 1,continu,40,0.591104
39,f_55,Zonnestraling,omgeving,continu,38,0.561549
33,f_41,temperatuur vertrek verwarmingsbatterij,luchtgroep 2,continu,4,0.059110
34,f_42,temperatuur retour verwarmingsbatterij,luchtgroep 2,continu,3,0.044333
0,f_16,totaal warmtegebruik warm,collector,continu,0,0.000000
1,f_17,totaal debiet,collector,continu,0,0.000000


In [82]:
# 2.2b - Alleen features met HEEL WEINIG spikes selecteren
MAX_SPIKE_PCT = 0.10   # max 0.50% van tijdstappen
MAX_SPIKE_N = 12       # én absoluut max 12 punten

sparse_spike_features = outlier_overzicht.loc[
    (outlier_overzicht["spike_n"] > 0) &
    (outlier_overzicht["spike_pct"] <= MAX_SPIKE_PCT) &
    (outlier_overzicht["spike_n"] <= MAX_SPIKE_N),
    "feature"
].tolist()

spike_fix_overzicht = outlier_overzicht[outlier_overzicht["feature"].isin(sparse_spike_features)].copy()

print("Aantal features met zeldzame extreme spikes:", len(sparse_spike_features))
display(spike_fix_overzicht.sort_values(["spike_n", "spike_pct"], ascending=False).head(50))

Aantal features met zeldzame extreme spikes: 2


,feature,beschrijving,systeem,signaal_type,spike_n,spike_pct
33,f_41,temperatuur vertrek verwarmingsbatterij,luchtgroep 2,continu,4,0.059110
34,f_42,temperatuur retour verwarmingsbatterij,luchtgroep 2,continu,3,0.044333


In [83]:
# 2.2c - Die zeldzame spikes wegwerken via interpolatie
data_2_2_clean = df_num.copy()

for f in sparse_spike_features:
    s = data_2_2_clean[f].mask(spike_mask[f])          # alleen gedetecteerde spikes naar NaN
    data_2_2_clean[f] = s.interpolate(method="time").ffill().bfill()

# Audit-info
mask_sparse_spikes = spike_mask[sparse_spike_features].astype("int8") if sparse_spike_features else pd.DataFrame(index=df_num.index)
print("Totaal vervangen spike-punten:", int(mask_sparse_spikes.sum().sum()) if len(sparse_spike_features) else 0)
print("Resterende missings:", int(data_2_2_clean.isna().sum().sum()))

Totaal vervangen spike-punten: 7
Resterende missings: 0


### Sectie 2.3: Analyse en statusvelden en binaire signalen op consistentie

In [84]:
src = data_2_2_clean.copy() if "data_2_2_clean" in globals() else (data_time_filled.copy() if "data_time_filled" in globals() else data.copy())
binaire_features = metadata_stats.loc[metadata_stats["signaal_type"] == "binair", "feature"].tolist()
bdf = src[binaire_features]

# Niet-binaire waarden opsporen (excl. NaN)
invalid_mask = ~(bdf.isna() | bdf.isin([0, 1]))
invalid_n = invalid_mask.sum()

# Dynamiek en balans
switch_rate_pct = 100 * bdf.diff().abs().fillna(0).mean()
pct_1 = 100 * (bdf == 1).mean()
missing_pct = 100 * bdf.isna().mean()

binaire_kwaliteit = pd.DataFrame({
    "feature": binaire_features,
    "n_missing": bdf.isna().sum().values,
    "missing_pct": missing_pct.values,
    "n_invalid_binary": invalid_n.values,
    "pct_1": pct_1.values,
    "switch_rate_pct": switch_rate_pct.values
}).merge(
    metadata_stats[["feature", "beschrijving", "systeem"]],
    on="feature",
    how="left"
)

# Eenvoudige flags + prioriteit
binaire_kwaliteit["flag_invalid"] = binaire_kwaliteit["n_invalid_binary"] > 0
binaire_kwaliteit["flag_imbalanced"] = (binaire_kwaliteit["pct_1"] < 1) | (binaire_kwaliteit["pct_1"] > 99)
binaire_kwaliteit["flag_flatline"] = binaire_kwaliteit["switch_rate_pct"] < 0.05

binaire_kwaliteit["prioriteit"] = (
    3 * binaire_kwaliteit["flag_invalid"].astype(int) +
    2 * binaire_kwaliteit["flag_flatline"].astype(int) +
    1 * binaire_kwaliteit["flag_imbalanced"].astype(int)
)

display(
    binaire_kwaliteit.sort_values(
        ["prioriteit", "n_invalid_binary", "switch_rate_pct"],
        ascending=[False, False, True]
    ).head(30)
)

,feature,n_missing,missing_pct,n_invalid_binary,pct_1,switch_rate_pct,beschrijving,systeem,flag_invalid,flag_imbalanced,flag_flatline,prioriteit
2,f_12,0,0.0,0,25.890350,1.315206,status pomp,gasketel(s),False,False,False,0
0,f_20,0,0.0,0,38.436530,2.497414,status pomp,collector,False,False,False,0
3,f_3,0,0.0,0,15.058372,6.723807,status,warmtepomp(en),False,False,False,0
4,f_7,0,0.0,0,21.146742,6.753362,status pomp,warmtepomp(en),False,False,False,0
1,f_10,0,0.0,0,15.294813,17.422787,status,gasketel(s),False,False,False,0


In [85]:
# Sectie 2.3b - cleanup binaire signalen (volledige cel)
src = data_2_2_clean.copy() if "data_2_2_clean" in globals() else (data_time_filled.copy() if "data_time_filled" in globals() else data.copy())
binaire_features = metadata_stats.loc[metadata_stats["signaal_type"] == "binair", "feature"].tolist()

data_2_3_clean = src.copy()  # <- ontbrak bij jou

if len(binaire_features) == 0:
    print("Geen binaire features om bij te werken.")
else:
    bdf = data_2_3_clean[binaire_features]

    # Maak niet-binaire waarden missing
    invalid_mask = ~(bdf.isna() | bdf.isin([0, 1]))
    data_2_3_clean.loc[:, binaire_features] = bdf.mask(invalid_mask)

    # Binary imputatie: ffill/bfill + afronden naar 0/1
    data_2_3_clean.loc[:, binaire_features] = (
        data_2_3_clean[binaire_features]
        .ffill()
        .bfill()
        .round()
        .clip(0, 1)
    )

    mask_binary_fix = invalid_mask.astype("int8")
    print("Aantal gecorrigeerde niet-binaire punten:", int(mask_binary_fix.sum().sum()))
    print("Resterende missings in binaire features:", int(data_2_3_clean[binaire_features].isna().sum().sum()))

Aantal gecorrigeerde niet-binaire punten: 0
Resterende missings in binaire features: 0


### Sectie 2.4: Harmonisatie en voorbereiding van data voor downstream preprocessing

In [86]:
# Sectie 2.4 - Geoptimaliseerde harmonisatie (behoudt fixes uit 2.2/2.3)
base = (
    data_2_3_clean.copy() if "data_2_3_clean" in globals()
    else data_2_2_clean.copy() if "data_2_2_clean" in globals()
    else data_time_filled.copy() if "data_time_filled" in globals()
    else data.copy()
)

# 1) Basis harmonisatie index
data_clean = base[~base.index.duplicated(keep="first")].sort_index()
freq_mode = data_clean.index.to_series().diff().dropna().mode().iloc[0]
volledige_index = pd.date_range(data_clean.index.min(), data_clean.index.max(), freq=freq_mode, tz=data_clean.index.tz)
data_clean = data_clean.reindex(volledige_index)

# 2) Type-informatie
type_map = metadata_stats.set_index("feature")["signaal_type"].to_dict()
binaire_features = [c for c in data_clean.columns if type_map.get(c) == "binair"]
discreet_features = [c for c in data_clean.columns if type_map.get(c) == "discreet"]
continue_features = [c for c in data_clean.columns if type_map.get(c) == "continu"]

# 3) Maskers
mask_missing_reindex = data_clean.isna().astype("int8")

# Reeds gecorrigeerde outliers uit 2.2 als masker meenemen (indien aanwezig)
if "mask_sparse_spikes" in globals():
    mask_spikes = mask_sparse_spikes.reindex(data_clean.index).fillna(0).astype("int8")
else:
    mask_spikes = pd.DataFrame(0, index=data_clean.index, columns=data_clean.columns, dtype="int8")

# Reeds gecorrigeerde binaire invalids uit 2.3 als masker meenemen (indien aanwezig)
if "mask_binary_fix" in globals():
    tmp = pd.DataFrame(0, index=data_clean.index, columns=data_clean.columns, dtype="int8")
    cols = [c for c in mask_binary_fix.columns if c in tmp.columns]
    tmp.loc[:, cols] = mask_binary_fix.reindex(data_clean.index)[cols].fillna(0).astype("int8")
    mask_binary_fix_full = tmp
else:
    mask_binary_fix_full = pd.DataFrame(0, index=data_clean.index, columns=data_clean.columns, dtype="int8")

# 4) Voorzichtig nulmasker enkel voor fysisch verdachte continue signalen
beschrijving_lc = metadata_stats.set_index("feature")["beschrijving"].astype(str).str.lower()
kandidaat_zero = [
    c for c in continue_features
    if (
        ("temperatuur" in beschrijving_lc.get(c, "") or "co2" in beschrijving_lc.get(c, "") or "temp" in beschrijving_lc.get(c, ""))
        and not any(k in beschrijving_lc.get(c, "") for k in ["setpunt", "gevraag", "omgeving", "buiten"])
    )
]
zero_pct = (data_clean[kandidaat_zero] == 0).mean() * 100 if len(kandidaat_zero) > 0 else pd.Series(dtype=float)
zero_fix_features = zero_pct[zero_pct <= 0.5].index.tolist()   # enkel zeldzame 0-waarden automatisch fixen

mask_zero_suspect = pd.DataFrame(0, index=data_clean.index, columns=data_clean.columns, dtype="int8")
if len(zero_fix_features) > 0:
    zero_cond = data_clean[zero_fix_features].eq(0)
    mask_zero_suspect.loc[:, zero_fix_features] = zero_cond.astype("int8")
    data_clean.loc[:, zero_fix_features] = data_clean[zero_fix_features].mask(zero_cond)

# 5) Type-specifieke imputatie
if len(continue_features) > 0:
    data_clean.loc[:, continue_features] = data_clean[continue_features].interpolate(method="time").ffill().bfill()

if len(discreet_features) > 0:
    data_clean.loc[:, discreet_features] = data_clean[discreet_features].ffill().bfill()

if len(binaire_features) > 0:
    data_clean.loc[:, binaire_features] = (
        data_clean[binaire_features].ffill().bfill().round().clip(0, 1)
    )

# 6) Finale maskers en checks
imputation_mask = (
    (mask_missing_reindex + mask_spikes + mask_binary_fix_full + mask_zero_suspect) > 0
).astype("int8")

print("Schone vorm:", data_clean.shape)
print("Resterende missings:", int(data_clean.isna().sum().sum()))
print("Aantal zero-fix features:", len(zero_fix_features))
print("Aantal gemarkeerde imputaties:", int(imputation_mask.sum().sum()))

Schone vorm: (6767, 54)
Resterende missings: 0
Aantal zero-fix features: 0
Aantal gemarkeerde imputaties: 7


In [87]:
# Korte validatie
print("Index regelmatig:", data_clean.index.to_series().diff().dropna().nunique() == 1)
print("Aantal binaire features met niet {0,1}:", int((~(data_clean[binaire_features].isin([0,1]) | data_clean[binaire_features].isna())).sum().sum()) if len(binaire_features) > 0 else 0)

Index regelmatig: True
Aantal binaire features met niet {0,1}: 0


------

### Sectie 3: Univariate tijdreeksanalyse

1. Evolutie van individuele signalen doorheen de tijd
2. Verdelingsanalyse per feature
3. Basale variabiliteit en stabiliteit per sensor
4. Dagelijkse patronen binnen de beschikbare periode

In [88]:
# Sectie 3.0 - Voorbereiding helpers en sets
df = data_clean.copy()

meta = metadata_stats.copy()

# Zorg dat feature altijd als index beschikbaar is
if "feature" in meta.columns:
    meta = meta.set_index("feature")
else:
    # fallback als feature al index was in eerdere run
    meta.index.name = "feature"

# Reindex op df-kolommen en herstel expliciet indexnaam
meta = meta.reindex(df.columns)
meta.index.name = "feature"
meta = meta.reset_index()

# Veiligheidscheck
assert meta["feature"].tolist() == df.columns.tolist(), "Metadata en df-kolommen niet uitgelijnd"

# Type- en systeemmappen
feature_to_system = meta.set_index("feature")["systeem"].fillna("onbekend").to_dict()
feature_to_type = meta.set_index("feature")["signaal_type"].fillna("onbekend").to_dict()

# Variabiliteitsscore voor ranking
var_score = df.std().sort_values(ascending=False)

# Parameters voor schaalbaarheid
TOP_N_CORR = 30
TOP_K_PER_SYSTEM = 4
MAX_SYSTEMS_PLOT = 12

print("Aantal features:", df.shape[1])
print("Aantal tijdstappen:", df.shape[0])
print("Aantal systemen:", meta["systeem"].nunique())

Aantal features: 54
Aantal tijdstappen: 6767
Aantal systemen: 8


### Sectie 3.1: Evolutie van individuele signalen doorheen de tijd

Tijdreeksen in HVAC-systemen (zoals ventilatoren of kleppen die tussen 0% en 100% schakelen) volgen zelden een normale verdeling. Statistische methodes zoals Z-scores falen hierop omdat ze "aan/uit" gedrag onterecht als extreme afwijkingen of wiskundige fouten (deling door nul) behandelen. 

Daarom gebruiken we hier een robuuste, niet-parametrische methode: **Kwantielen**.
Voor elk continu signaal berekenen we de grens waartussen **99% van de data** zich bevindt (tussen het 0,5% en 99,5% percentiel). Alles wat buiten deze grenzen valt, markeren we visueel als potentiële anomalie (ruis, sensorfouten of extreme pieken). De grafieken zijn gegroepeerd per systeem voor een intuïtieve controle.

In [ ]:
# Selecteer enkel de continue features en koppel ze aan hun systeem
continue_features = metadata_stats.loc[metadata_stats["signaal_type"] == "continu", "feature"].tolist()
feature_to_system = metadata_stats.set_index("feature")["systeem"].to_dict()

# Groepeer de features per systeem
systemen_dict = {}
for col in continue_features:
    sys = feature_to_system.get(col, 'onbekend')
    if sys not in systemen_dict:
        systemen_dict[sys] = []
    systemen_dict[sys].append(col)

print(f"Start genereren van visualisaties voor {len(systemen_dict)} systemen...")
print("Rode stippen duiden de 1% meest extreme waarden aan (0.5% ondergrens, 99.5% bovengrens).")

# Genereer plots per systeem
for systeem, features in systemen_dict.items():
    aantal_plots = len(features)
    # Maak de figure hoog genoeg op basis van het aantal features
    fig, axes = plt.subplots(aantal_plots, 1, figsize=(16, 3.5 * aantal_plots), sharex=True)
    
    if aantal_plots == 1:
        axes = [axes]
        
    fig.suptitle(f"Systeem: {systeem} - 99% Data Range", fontsize=16, fontweight='bold', y=1.02)
    
    for ax, col in zip(axes, features):
        serie = data_clean[col].dropna()
        if serie.empty:
            continue
            
        # Bereken de kwantielen voor 99% van de data
        p_low = serie.quantile(0.005)
        p_high = serie.quantile(0.995)
        
        # Plot het basissignaal
        ax.plot(serie.index, serie, color='#2980b9', alpha=0.7, linewidth=1.5, label='Signaal')
        
        # Plot de horizontale stippellijnen (grenzen)
        ax.axhline(p_high, color='#2c3e50', linestyle='--', alpha=0.8, label=f'99.5% ({p_high:.2f})')
        ax.axhline(p_low, color='#2c3e50', linestyle='--', alpha=0.8, label=f'0.5% ({p_low:.2f})')
        
        # Vind en markeer de uitschieters
        outliers = serie[(serie > p_high) | (serie < p_low)]
        if not outliers.empty:
            ax.scatter(outliers.index, outliers, color='#e74c3c', s=20, zorder=5, 
                       label=f'Uitschieters ({len(outliers)})')
            
        # Opmaak van de individuele as
        beschrijving = metadata_stats.loc[metadata_stats['feature'] == col, 'beschrijving'].values[0]
        ax.set_title(f"{col} | {beschrijving}", fontsize=11)
        ax.set_ylabel("Waarde")
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        
    plt.tight_layout()
    plt.show()

#### Sectie 3.1b: Outliers gericht aanpakken met gebruikersregels

In deze sectie corrigeren we outliers op een **controleerbare en reproduceerbare** manier.

Werkwijze:

1. De gebruiker kiest features via een nummer (niet via kolomnaam).
2. Per feature wordt gekozen:
   - welke grens: upper, lower of both
   - welke cut-off waarde(n)
   - welke methode: interpolate, clip of zero
3. De regels worden gevalideerd.
4. Daarna worden de correcties toegepast.
5. We eindigen met:
   - een audit-tabel
   - een masker van aangepaste punten
   - een opgeschoonde dataset

Aanbevolen standaard:
- Gebruik interpolate voor zeldzame pieken.
- Gebruik clip voor harde fysieke grenzen.
- Gebruik zero alleen als 0 fysisch betekenisvol is.

In [ ]:
# =========================
# GEBRUIKERSINSTELLINGEN
# =========================

# Brondata voor outlier-correctie
OUTLIER_INPUT_DF = data_clean.copy()

# Enkel continue features meenemen
OUTLIER_ONLY_CONTINUOUS = True

# Toon max zoveel rijen van de featurecatalogus
OUTLIER_CATALOG_PREVIEW_ROWS = 80

# Regels door gebruiker in te vullen:
# - nr: nummer uit de catalogus
# - side: "upper" | "lower" | "both"
# - cutoff_low / cutoff_high: absolute grenswaarden
# - method: "interpolate" | "clip" | "zero"
OUTLIER_RULES = [
    # Voorbeelden (pas aan):
    # {"nr": 3,  "side": "upper", "cutoff_low": np.nan, "cutoff_high": 85.0, "method": "interpolate"},
    # {"nr": 11, "side": "lower", "cutoff_low": 5.0,    "cutoff_high": np.nan, "method": "clip"},
    # {"nr": 18, "side": "both",  "cutoff_low": 10.0,   "cutoff_high": 90.0, "method": "clip"},
    {"nr": 18, "side": "lower",  "cutoff_low": 22.0,    "method": "clip"},
    {"nr": 19, "side": "lower",  "cutoff_low": 20.0,    "method": "clip"},
    {"nr": 11, "side": "both",  "cutoff_low": 18.0,   "cutoff_high": 46.0, "method": "clip"},
    {"nr": 8, "side": "both",  "cutoff_low": 18.0,   "cutoff_high": 52.0, "method": "clip"},
    {"nr": 21, "side": "lower",  "cutoff_low": 20.0,   "method": "clip"},
    {"nr": 22, "side": "lower",  "cutoff_low": 18.5,   "method": "clip"},
    {"nr": 23, "side": "lower",  "cutoff_low": 19.51,   "method": "clip"},
    {"nr": 24, "side": "both",  "cutoff_low": 491.75,   "cutoff_high": 2939.50, "method": "clip"},
    {"nr": 30, "side": "lower",  "cutoff_low": 18.0,   "method": "clip"},
    {"nr": 31, "side": "lower",  "cutoff_low": 17.5,   "method": "clip"},
    {"nr": 32, "side": "lower",  "cutoff_low": 18.0,   "method": "clip"},
    {"nr": 33, "side": "lower",  "cutoff_low": 17.5,   "method": "clip"},
    {"nr": 34, "side": "lower",  "cutoff_low": 19.0,   "method": "clip"},
    {"nr": 41, "side": "lower",  "cutoff_low": 19.0,   "method": "clip"},
    {"nr": 42, "side": "lower",  "cutoff_low": 19.0,   "method": "clip"},
    {"nr": 2, "side": "upper",  "cutoff_high": 1.0, "method": "clip"},
    {"nr": 4, "side": "lower",  "cutoff_low": 17.0,    "method": "clip"},
    {"nr": 6, "side": "lower",  "cutoff_low": 17.0,    "method": "clip"},
    {"nr": 43, "side": "both",  "cutoff_low": 21.25,   "cutoff_high": 21.25, "method": "clip"},
    {"nr": 44, "side": "both",  "cutoff_low": 19.0,   "cutoff_high": 22.5, "method": "clip"},
    {"nr": 45, "side": "lower",  "cutoff_low": 400,   "method": "clip"},
    {"nr": 47, "side": "lower",  "cutoff_low": 19.5,   "method": "interpolate"},
    {"nr": 48, "side": "lower",  "cutoff_low": 19.0,    "method": "clip"},
    {"nr": 49, "side": "both",  "cutoff_low": 200,   "cutoff_high": 980, "method": "clip"},
]

# Veiligheidsdrempel: waarschuwing bij te veel aangepaste punten per feature
OUTLIER_WARN_PCT_PER_FEATURE = 2.0

#### Stap 1 - Maak featurecatalogus met nummers

We bouwen een tabel met alle features die in aanmerking komen, zodat de gebruiker eenvoudig met nummers kan werken.

In [ ]:
# Featurecatalogus opbouwen
if OUTLIER_ONLY_CONTINUOUS:
    _meta = metadata_stats.loc[
        metadata_stats["signaal_type"] == "continu",
        ["feature", "beschrijving", "systeem"]
    ].copy()
else:
    _meta = metadata_stats.loc[:, ["feature", "beschrijving", "systeem"]].copy()

# Haal het echte feature-nummer uit 'f_XX'
# vb: f_16 -> 16
_feature_catalog = _meta.copy()
_feature_catalog["nr"] = (
    _feature_catalog["feature"]
    .astype(str)
    .str.extract(r"^f_(\d+)$", expand=False)
    .astype("Int64")
)

# Controle: als een feature niet het patroon f_<nummer> volgt
bad = _feature_catalog[_feature_catalog["nr"].isna()]
if not bad.empty:
    raise ValueError(
        "Niet-standaard featurenaam gevonden (verwacht f_<nummer>): "
        + ", ".join(bad["feature"].tolist())
    )

# nr vooraan zetten en optioneel sorteren op echt feature-nummer
_feature_catalog["nr"] = _feature_catalog["nr"].astype(int)
_feature_catalog = _feature_catalog[["nr", "feature", "beschrijving", "systeem"]]
_feature_catalog = _feature_catalog.sort_values("nr").reset_index(drop=True)

print("Aantal features in catalogus:", len(_feature_catalog))
display(_feature_catalog.head(OUTLIER_CATALOG_PREVIEW_ROWS))

#### Stap 2 - Regels valideren en toepassen

De code controleert eerst de regels op fouten (onbestaande feature-nummers, ontbrekende cut-offs, ongeldige methodes) en voert daarna de correctie uit.

In [ ]:
def _validate_outlier_rules(rules_df, catalog_df):
    valid_side = {"upper", "lower", "both"}
    valid_method = {"interpolate", "clip", "zero"}

    if rules_df.empty:
        raise ValueError("Geen regels opgegeven in OUTLIER_RULES.")

    required_cols = {"nr", "side", "cutoff_low", "cutoff_high", "method"}
    missing_cols = required_cols - set(rules_df.columns)
    if missing_cols:
        raise ValueError(f"Ontbrekende kolommen in regels: {sorted(missing_cols)}")

    unknown_nr = sorted(set(rules_df["nr"]) - set(catalog_df["nr"]))
    if unknown_nr:
        raise ValueError(f"Onbekende feature-nummers: {unknown_nr}")

    bad_side = rules_df.loc[~rules_df["side"].isin(valid_side), "side"].unique().tolist()
    if bad_side:
        raise ValueError(f"Ongeldige side-waarden: {bad_side}")

    bad_method = rules_df.loc[~rules_df["method"].isin(valid_method), "method"].unique().tolist()
    if bad_method:
        raise ValueError(f"Ongeldige method-waarden: {bad_method}")

    for i, r in rules_df.iterrows():
        if r["side"] in {"lower", "both"} and pd.isna(r["cutoff_low"]):
            raise ValueError(f"Rij {i}: cutoff_low ontbreekt voor side={r['side']}")
        if r["side"] in {"upper", "both"} and pd.isna(r["cutoff_high"]):
            raise ValueError(f"Rij {i}: cutoff_high ontbreekt voor side={r['side']}")


def _apply_outlier_rules(df_in, rules_df, catalog_df):
    df_out = df_in.copy()
    nr_to_feature = catalog_df.set_index("nr")["feature"].to_dict()

    adjusted_mask = pd.DataFrame(False, index=df_out.index, columns=df_out.columns)
    audit_rows = []

    for _, r in rules_df.iterrows():
        feature = nr_to_feature[int(r["nr"])]
        s = df_out[feature]

        mask_low = pd.Series(False, index=s.index)
        mask_high = pd.Series(False, index=s.index)

        if r["side"] in {"lower", "both"}:
            mask_low = s < float(r["cutoff_low"])
        if r["side"] in {"upper", "both"}:
            mask_high = s > float(r["cutoff_high"])

        mask = mask_low | mask_high
        n_adj = int(mask.sum())
        pct_adj = 100.0 * n_adj / len(df_out)

        if n_adj > 0:
            if r["method"] == "interpolate":
                s2 = s.mask(mask).interpolate(method="time").ffill().bfill()
                df_out[feature] = s2
            elif r["method"] == "clip":
                lo = float(r["cutoff_low"]) if r["side"] in {"lower", "both"} else -np.inf
                hi = float(r["cutoff_high"]) if r["side"] in {"upper", "both"} else np.inf
                df_out[feature] = s.clip(lower=lo, upper=hi)
            elif r["method"] == "zero":
                s2 = s.copy()
                s2.loc[mask] = 0.0
                df_out[feature] = s2

            adjusted_mask.loc[:, feature] = adjusted_mask[feature] | mask

        audit_rows.append({
            "nr": int(r["nr"]),
            "feature": feature,
            "side": r["side"],
            "method": r["method"],
            "cutoff_low": r["cutoff_low"],
            "cutoff_high": r["cutoff_high"],
            "n_adjusted": n_adj,
            "pct_adjusted": pct_adj
        })

    audit_df = pd.DataFrame(audit_rows).sort_values("pct_adjusted", ascending=False).reset_index(drop=True)
    return df_out, adjusted_mask, audit_df


# Regels in DataFrame
rules_df = pd.DataFrame(OUTLIER_RULES)

# Valideren + toepassen
_validate_outlier_rules(rules_df, _feature_catalog)
data_3_1_outlier_clean, mask_3_1_outlier_adjusted, audit_3_1_outlier = _apply_outlier_rules(
    OUTLIER_INPUT_DF, rules_df, _feature_catalog
)

print("Outlier-correctie uitgevoerd.")
print("Totaal aangepaste punten:", int(mask_3_1_outlier_adjusted.sum().sum()))
display(audit_3_1_outlier)

#### Stap 3 - Kwaliteitscontrole en finale dataset

We doen een snelle controle en zetten daarna de opgeschoonde dataset klaar voor de volgende secties.

In [ ]:
# Hard stop bij te veel aanpassingen per feature
too_high = audit_3_1_outlier.loc[
    audit_3_1_outlier["pct_adjusted"] > OUTLIER_WARN_PCT_PER_FEATURE
]

if not too_high.empty:
    print("WAARSCHUWING: sommige features overschrijden de toegelaten drempel.")
    display(too_high)

    # Notebook stopt hier: data_clean wordt NIET overschreven
    raise ValueError(
        f"Stop: minstens 1 feature heeft meer dan {OUTLIER_WARN_PCT_PER_FEATURE}% aanpassingen. "
        "Pas OUTLIER_RULES of de drempel aan en run opnieuw."
    )

# Enkel als alles OK is, nemen we de opgeschoonde data over
data_clean = data_3_1_outlier_clean.copy()

print("Finale dataset klaar.")
print("Shape:", data_clean.shape)
print("Resterende missings:", int(data_clean.isna().sum().sum()))

In [ ]:
# Selecteer enkel de continue features en koppel ze aan hun systeem
continue_features = metadata_stats.loc[metadata_stats["signaal_type"] == "continu", "feature"].tolist()
feature_to_system = metadata_stats.set_index("feature")["systeem"].to_dict()

# Groepeer de features per systeem
systemen_dict = {}
for col in continue_features:
    sys = feature_to_system.get(col, 'onbekend')
    if sys not in systemen_dict:
        systemen_dict[sys] = []
    systemen_dict[sys].append(col)

print(f"Start genereren van visualisaties voor {len(systemen_dict)} systemen...")
print("Rode stippen duiden de 1% meest extreme waarden aan (0.5% ondergrens, 99.5% bovengrens).")

# Genereer plots per systeem
for systeem, features in systemen_dict.items():
    aantal_plots = len(features)
    # Maak de figure hoog genoeg op basis van het aantal features
    fig, axes = plt.subplots(aantal_plots, 1, figsize=(16, 3.5 * aantal_plots), sharex=True)
    
    if aantal_plots == 1:
        axes = [axes]
        
    fig.suptitle(f"Systeem: {systeem} - 99% Data Range", fontsize=16, fontweight='bold', y=1.02)
    
    for ax, col in zip(axes, features):
        serie = data_clean[col].dropna()
        if serie.empty:
            continue
            
        # Bereken de kwantielen voor 99% van de data
        p_low = serie.quantile(0.005)
        p_high = serie.quantile(0.995)
        
        # Plot het basissignaal
        ax.plot(serie.index, serie, color='#2980b9', alpha=0.7, linewidth=1.5, label='Signaal')
        
        # Plot de horizontale stippellijnen (grenzen)
        ax.axhline(p_high, color='#2c3e50', linestyle='--', alpha=0.8, label=f'99.5% ({p_high:.2f})')
        ax.axhline(p_low, color='#2c3e50', linestyle='--', alpha=0.8, label=f'0.5% ({p_low:.2f})')
        
        # Vind en markeer de uitschieters
        outliers = serie[(serie > p_high) | (serie < p_low)]
        if not outliers.empty:
            ax.scatter(outliers.index, outliers, color='#e74c3c', s=20, zorder=5, 
                       label=f'Uitschieters ({len(outliers)})')
            
        # Opmaak van de individuele as
        beschrijving = metadata_stats.loc[metadata_stats['feature'] == col, 'beschrijving'].values[0]
        ax.set_title(f"{col} | {beschrijving}", fontsize=11)
        ax.set_ylabel("Waarde")
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        
    plt.tight_layout()
    plt.show()

### Sectie 3.2: Verdelingsanalyse per feature

In [ ]:
# 3.2a - Correlatie op systeemniveau (overzichtelijk)
# ----------------------------------------------------
MIN_FEATURES_PER_SYSTEM = 2          # systemen met minstens 2 features
MAX_SYSTEMS_FOR_PLOT = 20            # max aantal systemen in heatmap
ANNOT_IF_MAX_ITEMS = 18              # enkel annoteren als matrix klein genoeg is

# 1) feature -> systeem
feature_to_system = metadata_stats.set_index("feature")["systeem"].fillna("onbekend").to_dict()

# 2) z-normaliseer per feature, zodat systemen vergelijkbaar zijn qua schaal
df_num = data_clean.select_dtypes(include=[np.number]).copy()
df_z = (df_num - df_num.mean()) / (df_num.std(ddof=0) + 1e-8)

# 3) maak systeemprofielen (gemiddelde van features binnen systeem)
systeem_cols = {}
for c in df_z.columns:
    s = feature_to_system.get(c, "onbekend")
    systeem_cols.setdefault(s, []).append(c)

# filter op minimum aantal features
systeem_cols = {s: cols for s, cols in systeem_cols.items() if len(cols) >= MIN_FEATURES_PER_SYSTEM}

# als er veel systemen zijn, behoud de grootste
if len(systeem_cols) > MAX_SYSTEMS_FOR_PLOT:
    systeem_cols = dict(
        sorted(systeem_cols.items(), key=lambda kv: len(kv[1]), reverse=True)[:MAX_SYSTEMS_FOR_PLOT]
    )

sys_df = pd.DataFrame({s: df_z[cols].mean(axis=1) for s, cols in systeem_cols.items()})
corr_sys = sys_df.corr()

plt.figure(figsize=(11, 9))
sns.heatmap(
    corr_sys,
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    annot=(corr_sys.shape[0] <= ANNOT_IF_MAX_ITEMS),
    fmt=".2f",
    square=True,
    cbar_kws={"label": "Pearson correlatie"}
)
plt.title("Sectie 3.2a - Correlatiematrix op systeemniveau")
plt.tight_layout()
plt.show()

# Extra: sterkste systeemkoppels
mask_u = np.triu(np.ones(corr_sys.shape), k=1).astype(bool)
sys_pairs = (
    corr_sys.where(mask_u)
    .stack()
    .reset_index(name="corr")
    .rename(columns={"level_0": "systeem_1", "level_1": "systeem_2"})
)
sys_pairs["abs_corr"] = sys_pairs["corr"].abs()
display(sys_pairs.sort_values("abs_corr", ascending=False).head(15))


# 3.2b - Compacte feature-correlatiematrix
# -----------------------------------------
TOP_N_CORR = 30
corr_features = df_num.std().sort_values(ascending=False).head(TOP_N_CORR).index.tolist()
corr_top = df_num[corr_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_top,
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    xticklabels=True, yticklabels=True,
    cbar_kws={"label": "Pearson correlatie"}
)
plt.title(f"Sectie 3.2b - Correlatiematrix top {TOP_N_CORR} meest variabele features")
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()


# 3.2c - Samenvatting sterkste correlatieparen (duidelijker dan grote heatmap)
# ---------------------------------------------------------------------------
mask_u = np.triu(np.ones(corr_top.shape), k=1).astype(bool)
corr_pairs = (
    corr_top.where(mask_u)
    .stack()
    .reset_index(name="corr")
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2"})
)
corr_pairs["abs_corr"] = corr_pairs["corr"].abs()

print("Top 10 positieve correlaties")
display(corr_pairs.sort_values("corr", ascending=False).head(10))

print("Top 10 negatieve correlaties")
display(corr_pairs.sort_values("corr", ascending=True).head(10))

print("Top 15 absolute correlaties")
display(corr_pairs.sort_values("abs_corr", ascending=False).head(15))

### Sectie 3.3: Basale variabiliteit en stabiliteit per sensor

In [ ]:
# Voor HVAC erg informatief: toont regime per uur en feature in 1 beeld.
TOP_N_DAILY = 20
feat_daily = var_score.head(TOP_N_DAILY).index.tolist()

tmp = df[feat_daily].copy()
tmp["uur"] = tmp.index.hour

hour_profile = tmp.groupby("uur").median(numeric_only=True).T  # feature x uur
# Normaliseer per feature voor vergelijkbaarheid van patronen
hour_profile_z = (hour_profile.sub(hour_profile.mean(axis=1), axis=0)
                              .div(hour_profile.std(axis=1) + 1e-8, axis=0))

plt.figure(figsize=(14, 10))
sns.heatmap(hour_profile_z, cmap="RdBu_r", center=0, cbar_kws={"label": "Binnen-feature z-score"})
plt.title(f"Sectie 3.3 - Dagpatroon-heatmap (mediaan per uur, top {TOP_N_DAILY} features)")
plt.xlabel("Uur van de dag")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

### Sectie 3.4: Dagelijkse patronen binnen de beschikbare periode

In [ ]:
# Gemiddeld dagpatroon per tijdstip (uur:minuut)
dagprofiel = df.copy()
dagprofiel["tijdstip"] = dagprofiel.index.strftime("%H:%M")
gemiddeld_per_tijdstip = dagprofiel.groupby("tijdstip").mean(numeric_only=True)

# Plot voor enkele features
features_dag = df.columns[:4]

fig, axes = plt.subplots(len(features_dag), 1, figsize=(14, 8), sharex=True)

for i, col in enumerate(features_dag):
    axes[i].plot(gemiddeld_per_tijdstip.index, gemiddeld_per_tijdstip[col], linewidth=1.2)
    axes[i].set_title(f"Gemiddeld dagpatroon - {col}")
    axes[i].grid(alpha=0.3)

# x-as leesbaar maken
stap = 12  # om de 2 uur bij 10-minuten resolutie
for ax in axes:
    ax.set_xticks(range(0, len(gemiddeld_per_tijdstip.index), stap))
    ax.set_xticklabels(gemiddeld_per_tijdstip.index[::stap], rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Extra: verschil weekdag vs weekend (indien beide aanwezig)
df_week = df.copy()
df_week["is_weekend"] = df_week.index.dayofweek >= 5
df_week["tijdstip"] = df_week.index.strftime("%H:%M")

weekprofiel = df_week.groupby(["is_weekend", "tijdstip"]).mean(numeric_only=True)

feature = df.columns[0]  # kies 1 feature om te tonen
plt.figure(figsize=(12, 4))
plt.plot(weekprofiel.loc[False].index, weekprofiel.loc[False][feature], label="Weekdag")
if True in weekprofiel.index.get_level_values(0):
    plt.plot(weekprofiel.loc[True].index, weekprofiel.loc[True][feature], label="Weekend")
plt.title(f"Dagprofiel weekdag vs weekend - {feature}")
plt.xticks(range(0, len(weekprofiel.loc[False].index), 12), weekprofiel.loc[False].index[::12], rotation=45)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

----

## Sectie 4: Multievariabele relaties en systeemgedrag

1. Correlaties tussen relevante variabelen
2. Relaties binnen en tussen subsystemen
3. Lag- en trendindicaties tussen stuur- en responsvariabelen
4. Detectie van redundantie en potentiele featurecompressie

### Sectie 4.1: Correlaties tussen relevante variabelen

In [ ]:
# We nemen de opgeschoonde data
df4 = data_clean.copy()

# Kies de 20 meest variabele features (maakt de heatmap leesbaar)
top_n = 20
top_features = df4.std().sort_values(ascending=False).head(top_n).index.tolist()

corr_top = df4[top_features].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_top, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title(f"Correlatiematrix (top {top_n} meest variabele features)")
plt.tight_layout()
plt.show()

In [ ]:
# We werken met numerieke kolommen
num_df = df.select_dtypes(include=[np.number])

# Neem de meest variabele features voor een leesbare heatmap
top_features = num_df.std().sort_values(ascending=False).head(20).index
corr_top = num_df[top_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_top, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlatiematrix (top 20 meest variabele features)")
plt.tight_layout()
plt.show()

# Sterkste correlaties (zonder diagonaal)
corr_long = corr_top.where(np.triu(np.ones(corr_top.shape), k=1).astype(bool)).stack()
sterk_pos = corr_long.sort_values(ascending=False).head(10)
sterk_neg = corr_long.sort_values(ascending=True).head(10)

print("Top 10 sterkste positieve correlaties:")
print(sterk_pos)
print("\nTop 10 sterkste negatieve correlaties:")
print(sterk_neg)

In [ ]:
# Sterkste correlatieparen (zonder duplicaten)
corr_abs = corr_top.abs()
mask_upper = np.triu(np.ones(corr_abs.shape), k=1).astype(bool)

sterke_paren = (
    corr_abs.where(mask_upper)
    .stack()
    .sort_values(ascending=False)
)

sterke_paren.head(15)

### Sectie 4.2: Relaties binnen en tussen subsystemen

In [ ]:
# Feature -> systeem afleiden uit f_map
feature_meta = pd.DataFrame({"feature": df4.columns})
feature_meta["label"] = feature_meta["feature"].map(f_map).fillna("onbekend - onbekend")
feature_meta["label_eerste"] = feature_meta["label"].str.split(" | ").str[0]
feature_meta["systeem"] = feature_meta["label_eerste"].str.split(" - ", n=1).str[1].fillna("onbekend")

# Normaliseer per feature, zodat systemen vergelijkbaar worden
df4_z = (df4 - df4.mean()) / (df4.std(ddof=0) + 1e-8)

# Gemiddeld profiel per systeem doorheen de tijd
systemen = feature_meta["systeem"].dropna().unique().tolist()
systeem_profielen = {}

for sys in systemen:
    cols = feature_meta.loc[feature_meta["systeem"] == sys, "feature"].tolist()
    cols = [c for c in cols if c in df4_z.columns]
    if len(cols) > 0:
        systeem_profielen[sys] = df4_z[cols].mean(axis=1)

systeem_df = pd.DataFrame(systeem_profielen)
systeem_df.head()

In [ ]:
# Visualiseer systeemprofielen (rollend gemiddelde voor leesbaarheid)
plot_df = systeem_df.rolling(6, min_periods=1).mean()  # 6 stappen bij 10-min = 1 uur

plt.figure(figsize=(14, 6))
for col in plot_df.columns:
    plt.plot(plot_df.index, plot_df[col], label=col, linewidth=1)

plt.title("Genormaliseerd systeemgedrag doorheen de tijd (1u rolling mean)")
plt.grid(alpha=0.3)
plt.legend(loc="upper right", ncol=2)
plt.tight_layout()
plt.show()

### Sectie 4.3: Lag- en trendindicaties tussen stuur- en responsvariabelen

In [ ]:
# Kandidaten voor stuur- en responsvariabelen op basis van beschrijving
desc = feature_meta.set_index("feature")["label_eerste"].str.lower()

stuur_keywords = ["setpunt", "sturing", "klep", "ventiel", "status pomp", "gevraagde"]
respons_keywords = ["temperatuur", "debiet", "co2", "retour", "vertrek"]

stuur_features = [f for f in df4.columns if any(k in desc.get(f, "") for k in stuur_keywords)]
respons_features = [f for f in df4.columns if any(k in desc.get(f, "") for k in respons_keywords)]

print("Aantal stuurfeatures:", len(stuur_features))
print("Aantal responsfeatures:", len(respons_features))

In [ ]:
# Eenvoudige lag-analyse voor 1 stuur + 1 respons feature
# Je kan hier zelf andere kolommen kiezen uit stuur_features/respons_features
x_col = stuur_features[0]
y_col = respons_features[0]

max_lag = 12  # 12 stappen = 2 uur bij 10-min resolutie
lags = range(-max_lag, max_lag + 1)

lag_corr = []
for lag in lags:
    if lag < 0:
        corr = df4[x_col].corr(df4[y_col].shift(-lag))
    else:
        corr = df4[x_col].shift(lag).corr(df4[y_col])
    lag_corr.append(corr)

lag_df = pd.DataFrame({"lag": list(lags), "corr": lag_corr})

plt.figure(figsize=(10, 4))
plt.plot(lag_df["lag"], lag_df["corr"], marker="o")
plt.axhline(0, color="black", linewidth=0.8)
plt.title(f"Lag-correlatie: {x_col} -> {y_col}")
plt.xlabel("Lag (tijdstappen van 10 min)")
plt.ylabel("Correlatie")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

lag_df.loc[lag_df["corr"].abs().idxmax()]

### Sectie 4.4: Detectie van redundantie en potentiele feturecompressie

In [ ]:
# Redundantie op basis van zeer hoge absolute correlatie
corr_all = df4.corr().abs()
mask_upper = np.triu(np.ones(corr_all.shape), k=1).astype(bool)

redundante_paren = (
    corr_all.where(mask_upper)
    .stack()
    .reset_index()
)
redundante_paren.columns = ["feature_1", "feature_2", "abs_corr"]

# Drempel kan je aanpassen
threshold = 0.98
redundante_paren = redundante_paren[redundante_paren["abs_corr"] >= threshold].sort_values("abs_corr", ascending=False)

print(f"Aantal redundante paren (abs corr >= {threshold}): {len(redundante_paren)}")
redundante_paren.head(20)

In [ ]:
# Eenvoudige lijst met drop-kandidaten (telkens de 2e feature van elk paar)
drop_candidates = sorted(redundante_paren["feature_2"].unique().tolist())

print("Aantal unieke drop-kandidaten:", len(drop_candidates))
drop_candidates[:30]

------

## Sectie 5: Operationele segmentatie

1. Vergelijking werkuren versus niet-werkuren
2. Vergelijking weekdagen versus weekendprofielen
3. Clustering of groepering van typische bedrijfsmodi
4. Identificatie van contexten met verhoogde kans op afwijkingen

### Sectie 5.1: Vergelijking werkuren versus niet-werkuren

In [ ]:
# Basis dataframe voor sectie 5
df5 = data_clean.copy()

# Definieer werkuren: ma-vr, 08:00-18:00
is_weekdag = df5.index.dayofweek < 5
is_werkuur = (df5.index.hour >= 8) & (df5.index.hour < 18)

df5["segment_werkuren"] = np.where(is_weekdag & is_werkuur, "werkuren", "niet-werkuren")

# Gemiddelde per segment
segment_stats = df5.groupby("segment_werkuren").mean(numeric_only=True).T
segment_stats.head(20)

In [ ]:
# Visualiseer verschil voor enkele representatieve features
features_seg = df5.drop(columns=["segment_werkuren"]).std().sort_values(ascending=False).head(6).index

plot_df = (
    df5.groupby("segment_werkuren")[features_seg]
    .mean()
    .T
)

plot_df.plot(kind="bar", figsize=(12, 6))
plt.title("Gemiddelde waarde per feature: werkuren vs niet-werkuren")
plt.ylabel("Gemiddelde")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Sectie 5.2: Vergelijking weekdagen versus weekendprofielen

In [ ]:
df5_week = data_clean.copy()
df5_week["is_weekend"] = df5_week.index.dayofweek >= 5
df5_week["tijdstip"] = df5_week.index.strftime("%H:%M")

weekprofiel = df5_week.groupby(["is_weekend", "tijdstip"]).mean(numeric_only=True)

# Kies enkele features met hoge variatie
features_week = data_clean.std().sort_values(ascending=False).head(4).index

In [ ]:
fig, axes = plt.subplots(len(features_week), 1, figsize=(14, 10), sharex=True)

for i, col in enumerate(features_week):
    axes[i].plot(weekprofiel.loc[False].index, weekprofiel.loc[False][col], label="Weekdag")
    if True in weekprofiel.index.get_level_values(0):
        axes[i].plot(weekprofiel.loc[True].index, weekprofiel.loc[True][col], label="Weekend")
    axes[i].set_title(f"Weekdag vs weekend - {col}")
    axes[i].grid(alpha=0.3)
    axes[i].legend()

# x-as leesbaar maken
stap = 12  # om de 2 uur bij 10-minuten data
for ax in axes:
    ax.set_xticks(range(0, len(weekprofiel.loc[False].index), stap))
    ax.set_xticklabels(weekprofiel.loc[False].index[::stap], rotation=45)

plt.tight_layout()
plt.show()

### Sectie 5.3: Clustering of groepering van typische bedrijfsmodi

In [ ]:
# Downsample naar uurlijkse gemiddelden voor stabielere clustering
df_hour = data_clean.resample("1h").mean()

# Features schalen
X = StandardScaler().fit_transform(df_hour)

# Eenvoudige clustering (k=3 als startpunt)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)

df_hour_clustered = df_hour.copy()
df_hour_clustered["cluster"] = clusters

print("Aantal observaties per cluster:")
print(df_hour_clustered["cluster"].value_counts().sort_index())

In [ ]:
# Clusterprofielen (gemiddelde featurewaarden per cluster)
cluster_profielen = df_hour_clustered.groupby("cluster").mean(numeric_only=True).T
cluster_profielen.head(20)

In [ ]:
# Visualisatie op 2 features (hoogste variatie)
top2 = df_hour.std().sort_values(ascending=False).head(2).index

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_hour_clustered,
    x=top2[0],
    y=top2[1],
    hue="cluster",
    palette="tab10",
    s=30
)
plt.title("Typische bedrijfsmodi (clusterprojectie op 2 features)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Sectie 5.4: Identificatie van contexten met verhoogde kans op afwijkingen

In [ ]:
# Eenvoudige risicoscore op basis van robuuste z-score (median + MAD)
df_risk = data_clean.copy()

median = df_risk.median()
mad = (df_risk - median).abs().median() + 1e-8

robust_z = ((df_risk - median).abs() / mad)

# Gemiddelde afwijkingsscore over alle features per tijdstip
risk_score = robust_z.mean(axis=1)

# Drempel: top 1%
threshold = risk_score.quantile(0.99)
risk_flag = risk_score > threshold

print("Aantal hoge-risico tijdstappen:", risk_flag.sum())
print("Drempel (p99):", threshold)

In [ ]:
# Koppel risicoflags aan operationele context
context = pd.DataFrame(index=data_clean.index)
context["risk_score"] = risk_score
context["high_risk"] = risk_flag
context["is_weekend"] = context.index.dayofweek >= 5
context["uur"] = context.index.hour

# Risicopercentage per uur
risico_per_uur = context.groupby("uur")["high_risk"].mean() * 100
risico_per_uur

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(risico_per_uur.index, risico_per_uur.values, marker="o")
plt.title("Percentage high-risk tijdstappen per uur")
plt.xlabel("Uur van de dag")
plt.ylabel("High-risk (%)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

--------------

## Sectie 6: Voorbereiding op modellering

1. Selectie van definitieve features voor het tansformer-model
2. Motivatie voor normalisatie en windowing-keuze
3. Definieren van baselinegedrag voor latere anomaly-scoring
4. Samenvatting van beperkingen en implicaties voor interpretatie

### Sectie 6.1: Selectie van definitieve features voor het transformer-model

In [ ]:
# Start met data_clean en de informatie die we eerder hebben gegenereerd
df6 = data_clean.copy()

# 1) Verwijder redundante features op basis van drop_candidates
# (Deze hebben abs(corr) >= 0.98 en zijn geïdentificeerd in sectie 4.4)
features_after_redundancy = [c for c in df6.columns if c not in drop_candidates]

print(f"Features na verwijdering redundantie: {len(features_after_redundancy)}")
print(f"Features verwijderd: {len(drop_candidates)}")

# 2) Filter op basis van variabiliteit / informatiewaarde
# Features met zeer lage variabiliteit dragen weinig bij aan anomaly detection
cv_threshold = 0.01  # Coefficient of variation drempel

variabiliteit_all = (df6.std() / (df6.mean().abs() + 1e-8))
low_var_features = variabiliteit_all[variabiliteit_all < cv_threshold].index.tolist()
low_var_features = [f for f in low_var_features if f in features_after_redundancy]

print(f"\nFeatures met zeer lage variabiliteit (CV < {cv_threshold}): {len(low_var_features)}")
print(low_var_features[:10])

# 3) Finale feature list
features_final = [c for c in features_after_redundancy if c not in low_var_features]

print(f"\nFinale feature set: {len(features_final)} features")
print(f"Totale reductie: {len(data_clean.columns)} -> {len(features_final)} ({100 * (1 - len(features_final) / len(data_clean.columns)):.1f}% verwijderd)")

# Sla de finale feature list op
df_final = df6[features_final].copy()
df_final.head()

### Sectie 6.2: Motivatie voor normalisatie en windowing-keuze

In [ ]:
# 1) Analyse van feature scales (bereiken) om normalisatiekeuze te rechtvaardigen
feature_scales = pd.DataFrame({
    "min": df_final.min(),
    "max": df_final.max(),
    "range": df_final.max() - df_final.min(),
    "mean": df_final.mean(),
    "std": df_final.std()
})

# Features met zeer verschillende schalen benadrukken het belang van normalisatie
feature_scales["range_ratio"] = feature_scales["range"] / (feature_scales["range"].min() + 1e-8)

print("Top 10 features met grootste bereiksverschillen:")
print(feature_scales.sort_values("range_ratio", ascending=False).head(10))

# 2) Visualiseer schaalverschillen
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Linkergrafiek: originele schalen
for col in feature_scales.head(6).index:
    axes[0].plot(df_final[col], alpha=0.6, label=col)
axes[0].set_title("Originele schalen (verschillende magnitudes)")
axes[0].grid(alpha=0.3)
axes[0].legend()

# Rechtergrafiek: genormaliseerde schalen
df_normalized = (df_final - df_final.mean()) / (df_final.std() + 1e-8)
for col in feature_scales.head(6).index:
    axes[1].plot(df_normalized[col], alpha=0.6, label=col)
axes[1].set_title("Na StandardScaler normalisatie")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n✓ Normalisatie: StandardScaler per feature (mean=0, std=1) nodig voor transformer")

# 3) Windowing analyse: kies venstergrootte voor sequenties
# Afhankelijk van: temporale resolutie (10-min) en HVAC-dynamica
# Lage-frequente systemen (warmtepomp) hebben ~30min-1h stabilisatie
# Hoge-frequente systemen (luchtgroep) kunnen sneller veranderen

window_options = {
    "12 stappen": 120,      # 2 uur
    "24 stappen": 240,      # 4 uur
    "36 stappen": 360,      # 6 uur
    "48 stappen": 480,      # 8 uur
}

print("\nVenstergrootte-opties voor transformer:")
for desc, mins in window_options.items():
    print(f"  {desc}: {mins} minuten (real-world interval)")

print("\n✓ Aanbeveling: 24-36 stappen (4-6 uur) = goede balans tussen:")
print("  - Voldoende context voor patronen (HVAC-cycles)")
print("  - Beperkte rekenkosten")
print("  - Goed gedocumenteerde transformers in anomaly detection literatuur")

### Sectie 6.3: Definieren van baselinegedrag voor latere anomaly-scoring

In [ ]:
# 1) Baseline per operationele segment (werktijd vs niet-werktijd, mode-cluster)
df_baseline = data_clean[features_final].copy()
df_baseline["is_werkuur"] = (df_baseline.index.dayofweek < 5) & \
                            ((df_baseline.index.hour >= 8) & (df_baseline.index.hour < 18))
df_baseline["uur"] = df_baseline.index.hour

# 2) Bereken baseline quantiles per segment
baselines = {}

for segment_name, mask in [("werkgebied", df_baseline["is_werkuur"]), 
                           ("niet_werkgebied", ~df_baseline["is_werkuur"])]:
    segment_data = df_baseline.loc[mask, features_final]
    
    baselines[segment_name] = {
        "mean": segment_data.mean(),
        "std": segment_data.std(),
        "q05": segment_data.quantile(0.05),
        "q95": segment_data.quantile(0.95),
        "mad": (segment_data - segment_data.median()).abs().median()
    }
    
    print(f"\nBaseline statistieken - {segment_name} ({mask.sum()} samples):")
    print(baselines[segment_name]["mean"].head(5))

# 3) Visualiseer baseline ranges per segment
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (segment_name, mask) in enumerate([("werkgebied", df_baseline["is_werkuur"]), 
                                             ("niet_werkgebied", ~df_baseline["is_werkuur"])]):
    segment_data = df_baseline.loc[mask, features_final]
    
    # Boxplot op genormaliseerde schaal
    z_scores = (segment_data - segment_data.mean()) / (segment_data.std() + 1e-8)
    
    sns.boxplot(data=z_scores.iloc[:, :10], ax=axes[idx], showfliers=False)
    axes[idx].set_title(f"Baseline range - {segment_name}")
    axes[idx].set_ylabel("Genormaliseerde waarde (z-score)")
    axes[idx].grid(alpha=0.2)

plt.tight_layout()
plt.show()

# 4) Sla baselines op voor latere anomaly-scoring
print("\n✓ Baselines opgeslagen per segment; kan gebruikt worden voor:")
print("  - Threshold-bepaling voor anomaly-flags")
print("  - Context-aware anomaly-scoring (verschillend per bedrijfsmode)")
print("  - Validatie van anomalieën tegen normaal gedrag")

### Sectie 6.4: Samenvatting van beperkingen en implicaties voor interpretaties

In [ ]:
# 1) Samenvatting EDA-resultaten
print("=" * 70)
print("SAMENVATTING EDA-RESULTATEN VOOR ANOMALY DETECTION")
print("=" * 70)

summary = f"""
📊 DATASET KARAKTERISTIEKEN
  - Meetperiode: {data_clean.index.min().date()} tot {data_clean.index.max().date()}
  - Duratie: {(data_clean.index.max() - data_clean.index.min()).days} dagen (~1 maand)
  - Temporale resolutie: 10 minuten
  - Totaal samples: {len(data_clean):,}
  - Totaal features na reductie: {len(features_final)} (oorspronkelijk: {len(data_clean.columns)})

📈 DATA KWALITEIT
  - Ontbrekende waarden na opschoning: 0%
  - Interpolatiemethode: time-based lineair
  - Redundante features verwijderd: {len(drop_candidates)}
  - Lage-variabiliteit features verwijderd: {len(low_var_features)}

🔄 OPERATIONELE SEGMENTATIE
  - Werkgebied (ma-vr 08:00-18:00): {(df_baseline["is_werkuur"]).sum():,} samples
  - Niet-werkgebied: {(~df_baseline["is_werkuur"]).sum():,} samples
  - K-means clusters geïdentificeerd: 3 (modes)
  - High-risk contexten (p99 robust z-score): {(risk_score > risk_score.quantile(0.99)).sum():,} samples

🎯 MODELLERINGSVOORBEREIDING
  - Normalisatie: StandardScaler (per feature μ=0, σ=1)
  - Aanbevolen window-size: 24-36 stappen (4-6 uur)
  - Baseline-definitie: per operationele segment
  - Anomaly-thresholds: adaptief per segment (niet globaal)
"""

print(summary)

# 2) Beperkingen en voorzichtigheden
print("⚠️  BEPERKINGEN & VOORZICHTIGHEDEN")
print("-" * 70)

limitations = """
1. SEIZONALITEIT NIET WAARNEEMBAAR
   → Dataset omvat slechts ~1 maand; seizoenspatronen (zomer/winter) ontbreken
   → Model kan slecht generaliseren naar andere seizoenen
   → Aanbeveling: retrain elk halfjaar met nieuwere data

2. OPERATIONELE SHIFT MOGELIJK
   → Bedrijfsregels/instellingen kunnen per dag veranderen
   → Baseline gedefinieerd op 1 maand; kan verouderd raken
   → Aanbeveling: monitor baseline-drift met sliding window

3. EXTREME WAARDEN GECLIPPED
   → Outliers in 1-99% percentiel liggen; echte fouten mogelijk gemaskeerd
   → Robust z-score helpt, maar aanname: normale bedrijfsruis vs echte fout
   → Aanbeveling: post-processing van anomalies met domeinkennis

4. FEATURE-ONTLEDING SIMPEL
   → Redundantie bepaald op abs(corr) >= 0.98; arbitraire drempel
   → Sommige samenhang kan nuttig zijn voor context (bv. gekoppelde sensors)
   → Aanbeveling: experimenteren met lagere drempel (0.95) bij slechte prestatie

5. KORTE TRAININGSHORIZON
   → Slechts 1 maand data; rare-event-anomalies kunnen ontbreken
   → Transformer traint mogelijk op subset normale modi
   → Aanbeveling: ensemble met rule-based checks; interpreteer model uit voorzichtig

6. DOMEINKENNISBRONNEN NIET BESCHIKBAAR
   → Geen expert-labels van anomalieën gedurende meetperiode
   → Validatie uitsluitend via reconstructie-error of unsupervised metriek
   → Aanbeveling: na training met domain expert valideren

3. STUUR- EN RESPONSFEATURES NIET STRIKT CAUSAAL
   → Correlaties gevonden; maar causale richting niet ondubbelzinnig vastgesteld
   → Lag-analyse helpt, maar vereist meer data voor robuustheid
   → Aanbeveling: fysisch model consulteren (HVAC-ontwerpdocs)
"""

print(limitations)

# 3) Volgende stappen
print("\n" + "=" * 70)
print("VOLGENDE STAPPEN")
print("=" * 70)

next_steps = """
✅ EDA COMPLEET
   ✓ Data opgeschoond en gevalideerd
   ✓ Features geselecteerd en gereduceerd
   ✓ Baselinegedrag per segment gedefinieerd
   ✓ Operationele contexten gekarteerd

🚀 KLAAR VOOR MODELING
   → Transformer Autoencoder trainen op df_final (genormaliseerd)
   → Window-size: 24-36 stappen
   → Anomaly-threshold: per-segment robust z-score (p99)
   → Evaluatie: testverzameling uit niet-werkuren (distributieverandering)

📋 VALIDATIE EN DEPLOYMENT
   → Cross-validate op werkdagen vs weekends (domain shift)
   → Test anomaly-types: stagnatie, spikes, graduelle drift, regime-verandering
   → Interpreteer top-k anomalieën met domeinexpert
   → Monitoren baseline-drift in productiestadium
"""

print(next_steps)
print("=" * 70)

### Sectie 7 - Feature engineering en export

Als laatste stap voegen we nog features toe aan onze dataset:

- `is_weekdag`
- `is_werkdag`
- `is_werkuur`
- `uur`

Omdat dit relevante info bevat voor de anomaliedetectie later bij de transformer.

In [ ]:
data_export = data_clean.copy()

data_export["is_weekdag"] = (data_export.index.dayofweek < 5).astype("int8")
data_export["is_weekend"] = (data_export.index.dayofweek >= 5).astype("int8")
data_export["is_werkuur"] = (
    (data_export.index.dayofweek < 5)
    & (data_export.index.hour >= 8)
    & (data_export.index.hour < 18)
).astype("int8")
data_export["uur"] = data_export.index.hour.astype("int8")

data_export.index.name = "timestamp"

output_path = f"processed/{GEBOUWNAAM}.csv"
data_export.to_csv(output_path, index=True)

print(f"Opgeslagen naar: {output_path}")
print("Shape:", data_export.shape)
print("Kolommen:", list(data_export.columns))

-----------